# Vanilla Gap-K% on StarCoder2-3B (Kaggle)
Enable a **GPU accelerator** and enable **Internet** for the initial model/dataset download, unless cached assets are attached as a private Kaggle Dataset. The first download needs time and several GB of disk. This notebook only orchestrates the installed `code_mia` package; it contains no attack or dataset-processing implementation.

In [18]:
from pathlib import Path
import json, os, subprocess
REPOSITORY_URL = "https://github.com/sayon29/code-mia-gapk.git"
REPO = Path("/kaggle/working/code-mia-gapk")
if not (REPO / "pyproject.toml").exists():
    if REPOSITORY_URL.startswith("REPLACE_WITH"):
        raise FileNotFoundError(f"Upload the repository to {REPO}, or replace REPOSITORY_URL with a real Git URL.")
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO)], check=True)
os.chdir(REPO)
print(REPO)

/kaggle/working/code-mia-gapk


In [19]:
os.environ.update({
    "HF_HOME": "/kaggle/working/hf-cache",
    "HUGGINGFACE_HUB_CACHE": "/kaggle/working/hf-cache/hub",
    "HF_DATASETS_CACHE": "/kaggle/working/hf-cache/datasets",
    "TOKENIZERS_PARALLELISM": "false",
})

In [20]:
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

Obtaining file:///kaggle/working/code-mia-gapk
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for code-mia-gapk (pyproject.toml): started
  Building editable for code-mia-gapk (pyproject.toml): finished with status 'done'
  Created wheel for code-mia-gapk: filename=code_mia_gapk-0.1.0-0.editable-py3-none-any.whl size=6550 sha256=877d0ddb4bf14b25a26690f96b87927318cd059c87862b63acff87808b6c00c1
  Stored in directory: /tmp/pip-ephem-wheel-cache-g89na7fj/wheels/1a/09/e8/48a29e17737756587f23eb77d1b9d6c09ff414f1c986e510a7
Successfu

CompletedProcess(args=['python', '-m', 'pip', 'install', '-e', '.'], returncode=0)

In [21]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path("/kaggle/working/code-mia-gapk")
SCRIPT = REPO / "scripts" / "inspect_environment.py"

print("Current directory:", Path.cwd())
print("Repository exists:", REPO.exists())
print("Script path:", SCRIPT)
print("Script exists:", SCRIPT.exists())

assert REPO.exists(), f"Repository not found: {REPO}"
assert SCRIPT.exists(), f"Environment script not found: {SCRIPT}"

os.chdir(REPO)

report_run = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=str(REPO),
    capture_output=True,
    text=True,
    check=False,
)

print("Return code:", report_run.returncode)
print("\nSTDOUT:")
print(report_run.stdout)
print("\nSTDERR:")
print(report_run.stderr)

if report_run.returncode != 0:
    raise RuntimeError(
        "Environment inspection failed. Read the STDERR printed above."
    )

environment = json.loads(report_run.stdout)

Current directory: /kaggle/working/code-mia-gapk
Repository exists: True
Script path: /kaggle/working/code-mia-gapk/scripts/inspect_environment.py
Script exists: True
Return code: 0

STDOUT:
{
  "platform": "Linux-6.18.48+-x86_64-with-glibc2.39",
  "operating_system": "Linux",
  "python_version": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "4.57.6",
    "datasets": "3.6.0",
    "huggingface-hub": "0.36.2",
    "nltk": "3.9.1",
    "numpy": "2.1.3",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.0"
  },
  "torch_version": "2.11.0+cu128",
  "cuda_available": true,
  "torch_cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu_count": 2,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "vram_bytes": 15636037632,
      "free_vram_bytes": 15526068224,
      "total_visible_vram_bytes": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "vram_bytes": 1563603763

In [22]:
import json
import subprocess

subprocess.run(["nvidia-smi"], check=False)

report_run = subprocess.run(
    ["python", "scripts/inspect_environment.py"],
    check=True,
    capture_output=True,
    text=True,
)

print(report_run.stdout)

environment = json.loads(report_run.stdout)

if not environment["internet_dns_available"]:
    model_hint = environment.get("model_snapshot_hint") or {}
    dataset_hint = environment.get("dataset_snapshot_hint") or {}

    if not (
        model_hint.get("exists")
        and dataset_hint.get("exists")
    ):
        raise RuntimeError(
            "Internet/DNS is unavailable and pinned model/dataset "
            "snapshots are not both present in the Hugging Face cache. "
            "Enable Kaggle Internet for the first download, or attach "
            "local model and dataset snapshots."
        )

Mon Oct  5 07:49:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
subprocess.run(["pytest", "-q"], check=True)

...................                                                      [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/l

CompletedProcess(args=['pytest', '-q'], returncode=0)

## Build and audit the frozen 20+20 smoke manifest

In [16]:
subprocess.run(["code-mia", "build-manifest", "--config", "configs/smoke.yaml"], check=True)
import json
print(json.dumps(json.load(open("outputs/smoke/dataset_audit.json")), indent=2))

Generating test split: 100%|██████████| 20000/20000 [00:00<00:00, 73961.16 examples/s]
ERROR: Insufficient eligible rows: members=0/20, non-members=0/20


CalledProcessError: Command '['code-mia', 'build-manifest', '--config', 'configs/smoke.yaml']' returned non-zero exit status 2.

In [ ]:
subprocess.run(["code-mia", "run", "--config", "configs/smoke.yaml"], check=True)

In [ ]:
subprocess.run(["code-mia", "validate-run", "--run-dir", "outputs/smoke"], check=True)
import pandas as pd
from IPython.display import Image, display
print(open("outputs/smoke/metrics.md").read())
display(pd.read_csv("outputs/smoke/scores.csv").head())
display(Image("outputs/smoke/roc_curve.png"), Image("outputs/smoke/score_histogram.png"))
scores = pd.read_csv("outputs/smoke/scores.csv")
print("Output:", (REPO / "outputs/smoke").resolve())
print("Peak allocated bytes:", scores.peak_allocated_gpu_bytes.max())
print("Summed sample runtime seconds:", scores.runtime_seconds.sum())

## Optional longer 500+500 debug run
Run this separately when ready. It is a debugging experiment, not a definitive scientific result.

In [ ]:
subprocess.run(["code-mia", "run", "--config", "configs/debug_gapk_starcoder2_3b.yaml"], check=True)

In [ ]:
DEBUG = REPO / "outputs/debug_gapk_starcoder2_3b"
subprocess.run(["code-mia", "validate-run", "--run-dir", str(DEBUG)], check=True)
print((DEBUG / "metrics.md").read_text())
display(pd.read_csv(DEBUG / "scores.csv").head())
display(Image(str(DEBUG / "roc_curve.png")), Image(str(DEBUG / "score_histogram.png")))

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/code-mia-gapk-results", "zip", REPO / "outputs")
print(archive)